# Segunda Semana - Entregable: Dataset Preprocesado y Notebook Documentado

## 🎯 Objetivo
Este notebook tiene como finalidad preparar el dataset para tareas de clasificación de imágenes del proyecto **FruitScan**. En él se abordan los siguientes pasos:

- Montaje de Google Drive para acceder a los datos.
- Aplicación de transformaciones a las imágenes (normalización, cambio de tamaño, etc.).
- Carga del dataset en PyTorch utilizando `ImageFolder`.
- División en conjuntos de entrenamiento y validación.
- Visualización de imágenes de ejemplo.

## Importación de librerías

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

## Configuración de rutas y parámetros

In [ ]:
data_dir = "/content/drive/MyDrive/FruitScan/data/fruits"
batch_size = 8
img_size = 224
val_split = 0.2
seed = 42

## Transformaciones
Se aplican transformaciones estándar que preparan las imágenes para su uso con modelos preentrenados:

In [ ]:
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Carga del dataset
Se utiliza ImageFolder, que asume que las subcarpetas de data_dir representan clases distintas:

In [ ]:
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
class_names = dataset.classes
print(f"Clases detectadas: {class_names}")

## División en entrenamiento y validación
Se divide el dataset usando una semilla fija para asegurar reproducibilidad:

In [ ]:
val_size = int(len(dataset) * val_split)
train_size = len(dataset) - val_size

torch.manual_seed(seed)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

## Creación de los DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## Visualización de imágenes del conjunto de entrenamiento

In [ ]:
def show_batch(dataloader):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(1, len(images), figsize=(15, 5))
    for img, label, ax in zip(images, labels, axes):
        img = img.permute(1, 2, 0) * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
        img = img.clip(0, 1)
        ax.imshow(img)
        ax.set_title(class_names[label])
        ax.axis('off')
    plt.show()

show_batch(train_loader)